## 0. Setup and imports

In this section we import TensorFlow and load the MNIST dataset.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension: (N, 28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

num_classes = 10
input_shape = x_train.shape[1:]

print("Train shape:", x_train.shape, "Test shape:", x_test.shape)

TensorFlow version: 2.19.0
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train shape: (60000, 28, 28, 1) Test shape: (10000, 28, 28, 1)


## 1. Practice: Basic tensors and operations

In this exercise, you will:

- Create random tensors
- Compute basic statistics with TensorFlow ops
- Reshape and slice tensors

### Task 1.1

Create a function `create_random_tensor(n)` that:

- Takes an integer `n`
- Returns a 1D tensor of shape `(n,)` with values sampled from a normal distribution `N(0, 1)` **using TensorFlow**

### Task 1.2

Create a function `describe_tensor(x)` that:

- Takes a 1D tensor `x`
- Prints:
  - shape
  - dtype
  - mean
  - standard deviation
  - min and max values

In [2]:
import tensorflow as tf

def create_random_tensor(n: int) -> tf.Tensor:
    """Return a 1D tensor of shape (n,) sampled from N(0, 1)."""
    x = tf.random.normal(shape=(n,), mean=0.0, stddev=1.0, seed=SEED)
    return x

def describe_tensor(x: tf.Tensor) -> None:
    """Print basic information about the tensor x."""
    print("Shape:", x.shape)
    print("Dtype:", x.dtype)
    mean = tf.reduce_mean(x)
    std = tf.math.reduce_std(x)
    min_v = tf.reduce_min(x)
    max_v = tf.reduce_max(x)
    print("Mean:", float(mean.numpy()))
    print("Std: ", float(std.numpy()))
    print("Min: ", float(min_v.numpy()))
    print("Max: ", float(max_v.numpy()))

# Try it
print("=== Task 1: Basic Tensors ===")
x = create_random_tensor(10)
describe_tensor(x)

# Additional practice: reshape and slice
print("\nReshape practice:")
x_2d = tf.reshape(x, (2, 5))
print("2D tensor shape:", x_2d.shape)
print("2D tensor:\n", x_2d.numpy())

print("\nSlice practice:")
print("First 3 elements:", x[0:3].numpy())

=== Task 1: Basic Tensors ===
Shape: (10,)
Dtype: <dtype: 'float32'>
Mean: 0.28105780482292175
Std:  0.6738396286964417
Min:  -0.7991440892219543
Max:  1.3148773908615112

Reshape practice:
2D tensor shape: (2, 5)
2D tensor:
 [[ 1.3148774  -0.15421568  0.9113878  -0.7991441  -0.10875294]
 [ 0.28436786  0.7661625  -0.6211289   0.9974318   0.21959257]]

Slice practice:
First 3 elements: [ 1.3148774  -0.15421568  0.9113878 ]


## 2. Practice: Dense neural network on MNIST

Now you will build a simple **fully-connected (dense) network** for MNIST classification.

### Task 2.1 — Build the model

Implement `build_dense_model(input_shape, num_classes)`:

- Input: `input_shape = (28, 28, 1)`
- Architecture (recommended):
  - `Flatten`
  - `Dense(256, activation="relu")`
  - `Dense(128, activation="relu")`
  - `Dense(num_classes, activation="softmax")`
- Compile with:
  - `Adam(1e-3)`
  - `"sparse_categorical_crossentropy"`
  - metric `"accuracy"`

### Task 2.2 — Train the model

Train the model for **5 epochs** with batch size 128 and use `validation_split=0.1`.

Observe:
- Training / validation accuracy
- Underfitting or overfitting?


In [3]:
def build_dense_model(input_shape, num_classes):
    """Build and compile a dense neural network for MNIST."""
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Flatten()(inputs)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

print("\n=== Task 2: Dense Neural Network ===")
dense_model = build_dense_model(input_shape, num_classes)
dense_model.summary()

history_dense = dense_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

print("\nDense model final validation accuracy:",
      history_dense.history["val_accuracy"][-1])


=== Task 2: Dense Neural Network ===


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9169 - loss: 0.2883 - val_accuracy: 0.9655 - val_loss: 0.1135
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9659 - loss: 0.1148 - val_accuracy: 0.9732 - val_loss: 0.0875
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9789 - loss: 0.0726 - val_accuracy: 0.9750 - val_loss: 0.0839
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9857 - loss: 0.0490 - val_accuracy: 0.9753 - val_loss: 0.0799
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9900 - loss: 0.0344 - val_accuracy: 0.9775 - val_loss: 0.0813

Dense model final validation accuracy: 0.9775000214576721


## 3. Practice: Convolutional Neural Network (CNN) on MNIST

Now build a small **CNN** which usually works better for images.

### Task 3.1 — Build a CNN model

Implement `build_cnn_model(input_shape, num_classes)` with:

- `Conv2D(32, kernel_size=3, activation="relu")`
- `MaxPooling2D()`
- `Conv2D(64, kernel_size=3, activation="relu")`
- `MaxPooling2D()`
- `Flatten`
- `Dense(128, activation="relu")`
- `Dense(num_classes, activation="softmax")`

Compile the model with the same settings as before.

### Task 3.2 — Train and compare

Train for **5 epochs** and compare validation accuracy with the dense model.


In [4]:
def build_cnn_model(input_shape, num_classes):
    """Build and compile a simple CNN for MNIST."""
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = keras.layers.MaxPooling2D()(x)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

print("\n=== Task 3: CNN ===")
cnn_model = build_cnn_model(input_shape, num_classes)
cnn_model.summary()

history_cnn = cnn_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128,
    verbose=1
)

print("\nCNN final validation accuracy:",
      history_cnn.history["val_accuracy"][-1])
print("\nComparison:")
print(f"Dense model val accuracy: {history_dense.history['val_accuracy'][-1]:.4f}")
print(f"CNN model val accuracy:   {history_cnn.history['val_accuracy'][-1]:.4f}")
print(f"Improvement:              {(history_cnn.history['val_accuracy'][-1] - history_dense.history['val_accuracy'][-1])*100:.2f} pp")


=== Task 3: CNN ===


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 57s 132ms/step - accuracy: 0.9448 - loss: 0.1898 - val_accuracy: 0.9823 - val_loss: 0.0663
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 60s 143ms/step - accuracy: 0.9836 - loss: 0.0522 - val_accuracy: 0.9877 - val_loss: 0.0471
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 56s 133ms/step - accuracy: 0.9890 - loss: 0.0345 - val_accuracy: 0.9868 - val_loss: 0.0461
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 55s 131ms/step - accuracy: 0.9921 - loss: 0.0251 - val_accuracy: 0.9860 - val_loss: 0.0453
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 83s 132ms/step - accuracy: 0.9945 - loss: 0.0178 - val_accuracy: 0.9890 - val_loss: 0.0403

CNN final validation accuracy: 0.9890000224113464

Comparison:
Dense model val accuracy: 0.9775
CNN model val accuracy:   0.9890
Improvement:              1.15 pp


## 4. Practice: `tf.data` input pipeline

In this section, you will build a `tf.data.Dataset` for MNIST and use it with `.fit()`.

### Task 4.1 — Create datasets

Implement `make_dataset(x, y, batch_size)` that:

- creates a `tf.data.Dataset` from `(x, y)`
- shuffles with buffer size `10000`
- batches with `batch_size`
- prefetches with `tf.data.AUTOTUNE`

Create:

- `train_ds` from `x_train, y_train`
- `test_ds` from `x_test, y_test` (without shuffling)

### Task 4.2 — Train with `tf.data`

Train the **CNN model** from the previous section using `train_ds` instead of NumPy arrays.


In [5]:
def make_dataset(x, y, batch_size: int, shuffle: bool = True) -> tf.data.Dataset:
    """Create a tf.data.Dataset with shuffle, batch, and prefetch."""
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=10000, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("\n=== Task 4: tf.data Pipeline ===")
batch_size = 128
train_ds = make_dataset(x_train, y_train, batch_size=batch_size, shuffle=True)
test_ds = make_dataset(x_test, y_test, batch_size=batch_size, shuffle=False)

# Verify dataset
print("Train dataset:", train_ds)
print("Test dataset:", test_ds)

# Rebuild a fresh CNN model for training with tf.data
cnn_model_ds = build_cnn_model(input_shape, num_classes)

history_cnn_ds = cnn_model_ds.fit(
    train_ds,
    epochs=5,
    validation_data=test_ds,
    verbose=1
)

print("\nCNN with tf.data final test accuracy:",
      history_cnn_ds.history["val_accuracy"][-1])


=== Task 4: tf.data Pipeline ===
Train dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>
Test dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.uint8, name=None))>
Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 63s 130ms/step - accuracy: 0.9414 - loss: 0.1963 - val_accuracy: 0.9792 - val_loss: 0.0569
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 62s 133ms/step - accuracy: 0.9836 - loss: 0.0538 - val_accuracy: 0.9833 - val_loss: 0.0483
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 80s 129ms/step - accuracy: 0.9883 - loss: 0.0385 - val_accuracy: 0.9859 - val_loss: 0.0401
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 84s 132ms/step - accuracy: 0.9915 - loss: 0.0279 - val_accuracy: 0.9889 - val_loss: 0.0313
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 61s 129ms/step - accuracy: 0.9929 - loss: 0.0218 - val_accuracy: 0.9907 

## 5. Practice: Custom training loop with `tf.GradientTape`

Here we will train a small network for a **toy regression problem** using a fully custom loop.

We will:

1. Generate synthetic data for `y = 3x + noise`
2. Build a small dense model with Keras
3. Write the training loop using `tf.GradientTape`

### Task 5.1 — Generate data

- Create 1000 points `x` uniformly in [-1, 1]
- Compute `y = 3 * x + noise`, where noise is normal with std 0.1

### Task 5.2 — Build model

- Simple `Sequential` model:
  - `Dense(8, activation="relu", input_shape=(1,))`
  - `Dense(1)`

### Task 5.3 — Custom loop

For several epochs:

- for each batch:
  - run forward pass
  - compute MSE loss
  - compute gradients w.r.t. trainable variables
  - apply gradients with Adam optimizer

Observe how loss decreases and how learned weight compares to 3.0.


In [6]:
print("\n=== Task 5: Custom Training Loop ===")

# 5.1 Generate data for regression y = 3x + noise
n_samples = 1000
x_reg = np.random.uniform(-1.0, 1.0, size=(n_samples, 1)).astype("float32")
noise = np.random.normal(loc=0.0, scale=0.1, size=(n_samples, 1)).astype("float32")
y_reg = 3.0 * x_reg + noise

# Build dataset
batch_size_reg = 32
reg_ds = tf.data.Dataset.from_tensor_slices((x_reg, y_reg))
reg_ds = reg_ds.shuffle(buffer_size=1000, seed=SEED).batch(batch_size_reg)

# 5.2 Build model
reg_model = keras.Sequential([
    keras.layers.Dense(8, activation="relu", input_shape=(1,)),
    keras.layers.Dense(1),
])

optimizer = keras.optimizers.Adam(learning_rate=1e-2)
loss_fn = keras.losses.MeanSquaredError()

# 5.3 Custom training loop
n_epochs_reg = 20

print("\nTraining with custom loop:")
for epoch in range(n_epochs_reg):
    epoch_loss = 0.0
    n_batches = 0

    for x_batch, y_batch in reg_ds:
        with tf.GradientTape() as tape:
            y_pred = reg_model(x_batch, training=True)
            loss_value = loss_fn(y_batch, y_pred)

        grads = tape.gradient(loss_value, reg_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, reg_model.trainable_variables))

        epoch_loss += float(loss_value.numpy())
        n_batches += 1

    epoch_loss /= n_batches
    print(f"Epoch {epoch+1:02d}: loss = {epoch_loss:.4f}")

# Inspect learned weight (should be close to 3.0)
print("\nLearned parameters:")
for var in reg_model.trainable_variables:
    print(f"  {var.name}: {var.numpy()}")

# Test prediction
test_x = np.array([[0.5], [-0.5], [1.0]], dtype=np.float32)
test_y = reg_model.predict(test_x)
print(f"\nTest predictions (true y = 3x):")
for i in range(len(test_x)):
    print(f"  x={test_x[i][0]:.1f}: predicted={test_y[i][0]:.2f}, actual={3*test_x[i][0]:.1f}")


=== Task 5: Custom Training Loop ===

Training with custom loop:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 01: loss = 1.5565
Epoch 02: loss = 0.1403
Epoch 03: loss = 0.0471
Epoch 04: loss = 0.0280
Epoch 05: loss = 0.0209
Epoch 06: loss = 0.0177
Epoch 07: loss = 0.0154
Epoch 08: loss = 0.0137
Epoch 09: loss = 0.0126
Epoch 10: loss = 0.0120
Epoch 11: loss = 0.0114
Epoch 12: loss = 0.0114
Epoch 13: loss = 0.0107
Epoch 14: loss = 0.0103
Epoch 15: loss = 0.0104
Epoch 16: loss = 0.0101
Epoch 17: loss = 0.0100
Epoch 18: loss = 0.0099
Epoch 19: loss = 0.0108
Epoch 20: loss = 0.0101

Learned parameters:
  kernel: [[ 0.34118623 -0.83084977  0.7468163  -0.2588862   0.5793191  -0.6747553
   0.55062646 -0.6719885 ]]
  bias: [-0.34121367  0.8155225   0.6706399  -0.29805437  0.56695     0.61561567
  0.53902364  0.66153336]
  kernel: [[-0.10591653]
 [-0.44448283]
 [ 0.534738  ]
 [ 0.4997501 ]
 [ 0.90255547]
 [-1.1135899 ]
 [ 0.80819833]
 [-0.80028033]]
  bias: [0.2938519]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step

Test predictions (true y = 3x):
  x=0.5: predicted=1.54, actual=1.5
  x=-0.5: predicted=-1.